## Autograd

#torch.autograd is PyTorch’s automatic differentiation engine that powers neural network training

In [1]:
from sympy.printing.pytorch import torch

"""
Training a NN happens in two steps:

Forward Propagation: In forward prop, the NN makes its best guess about the correct output. It runs the input data through each of its functions to make this guess.

Backward Propagation: In backprop, the NN adjusts its parameters proportionate to the error in its guess. It does this by traversing backwards from the
output, collecting the derivatives of the error with respect to the parameters of the functions (gradients), and optimizing the parameters using gradient
descent. For a more detailed walkthrough of backprop
"""

'\nTraining a NN happens in two steps:\n\nForward Propagation: In forward prop, the NN makes its best guess about the correct output. It runs the input data through each of its functions to make this guess.\n\nBackward Propagation: In backprop, the NN adjusts its parameters proportionate to the error in its guess. It does this by traversing backwards from the\noutput, collecting the derivatives of the error with respect to the parameters of the functions (gradients), and optimizing the parameters using gradient\ndescent. For a more detailed walkthrough of backprop\n'

In [2]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

In [3]:
model = resnet18(weights='ResNet18_Weights.DEFAULT')
data = torch.rand((1,3,64,64))
label = torch.rand(1,1000) # ResNet18 has 1000 classes

In [4]:
prediction = model(data)

In [5]:
loss = (prediction - label).sum() # In general, we do not use something like .sum() as loss function because at the end of forward pass we could get something extraordinary, Hence, it may cause a Gradient Exploding

loss.backward()

In [6]:
optim = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [7]:
optim.step() #gradient descent

So far, we done the basic NN application as follows:

Forward pass -> Produce a number -> Activation Functions (e.g. Tanh) -> Generate an loss using loss function -> Apply backpropagation via loss function -> Using optimizer apply Gradient Descent and adjust the weights.

## Differentiation in Autograd

In [8]:
import torch
a = torch.tensor([2., 3.], requires_grad=True) # it allows to follow tensor a when we typed requires_grad. (Andrej micrograd video)
b = torch.tensor([1., 2.], requires_grad=True)

Q = 3*a**3 - b**2 ## Attention: Q is not a scaler. Hence, we must enter a one matrix as DQ / DQ = 1. 

"""
DQ/Da =  9a**2
DQ/Db = -2*b
"""

# When we call .backward() on Q, autograd calculates these gradients and stores them in the respective tensors’ .grad attribute.

"""What I understand is this: The Jacobian is concerned with how $y$ changes with respect to $x$ values. To update the weights, we need the gradients. By multiplying the change of Loss with respect to Output ($\frac{\partial L}{\partial y}$) by the change of Output with respect to $x$ ($\frac{\partial y}{\partial x}$), we directly find the change of Loss with respect to $x$. Then, we update the weights using optimizers.
"""



<>:14: SyntaxWarning: invalid escape sequence '\p'
<>:14: SyntaxWarning: invalid escape sequence '\p'
C:\Users\calis\AppData\Local\Temp\ipykernel_12192\2907552523.py:14: SyntaxWarning: invalid escape sequence '\p'
  """What I understand is this: The Jacobian is concerned with how $y$ changes with respect to $x$ values. To update the weights, we need the gradients. By multiplying the change of Loss with respect to Output ($\frac{\partial L}{\partial y}$) by the change of Output with respect to $x$ ($\frac{\partial y}{\partial x}$), we directly find the change of Loss with respect to $x$. Then, we update the weights using optimizers.


'What I understand is this: The Jacobian is concerned with how $y$ changes with respect to $x$ values. To update the weights, we need the gradients. By multiplying the change of Loss with respect to Output ($\x0crac{\\partial L}{\\partial y}$) by the change of Output with respect to $x$ ($\x0crac{\\partial y}{\\partial x}$), we directly find the change of Loss with respect to $x$. Then, we update the weights using optimizers.\n'

In [9]:
## The output tensor of an operation will require gradients even if only a single input tensor has requires_grad=True

x = torch.rand(5,5)
y = torch.rand(5,5)
z = torch.rand((5,5), requires_grad = True)

a = x + y
print(f"Does 'a' require grad ?: {a.requires_grad}")
b = x+z
print(f"Does 'b' require grad ?: {b.requires_grad}")

Does 'a' require grad ?: False
Does 'b' require grad ?: True


In [10]:
# In finetuning, we freeze most of the model and typically only modify the classifier layers to make predictions on new labels
# Let's walk through a small example:


model = resnet18(weights=ResNet18_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False

In [11]:
print(model) # via printing model we can see the last classifier layer.

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [12]:
#(fc): is our decision layer.

model.fc = nn.Linear(512, 10)

#Now all parameters in the model, except the parameters of model.fc, are frozen. The only parameters that compute gradients are the weights and bias of model.fc

# Optimize only the classifier
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)